In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [11]:
#Read the CSV, remove any whitespace from the columns and print the first 10 lines
df = pd.read_csv("E:\\Project Portfolio\\Dissertation\\Final-Year-IDS\\data\\Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv", skipinitialspace=True)
df.head(10)

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,54865,3,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
1,55054,109,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
2,55055,52,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
3,46236,34,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
4,54863,3,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
5,54871,1022,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
6,54925,4,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
7,54925,42,1,1,6,6,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
8,9282,4,2,0,12,0,6,6,6.0,0.00000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN
9,55153,4,2,0,37,0,31,6,18.5,17.67767,...,20,0.0,0.0,0,0,0.0,0.0,0,0,BENIGN


In [12]:
#Creating input features for the ML model.  Dropping unnecessary input columns for the X and making the Y boolean as packets are either Safe or unsafe
y = (df["Label"] !="BENIGN").astype(int)
print("Label Distribution (0=Safe, 1=unsafe): ")
print(y.value_counts())

#These columns will be dropped as they are not needed and/or create unnecessary noise which can cause overfitting
drop_cols = ["Label","Flow ID", "Source IP", "Destination IP", "Timestamp"]
x = df.drop(columns=[c for c in drop_cols if c in df.columns])

#feature matix shape
print("Feature matix:", x.shape)
x.isna().sum().sum()

Label Distribution (0=Safe, 1=unsafe): 
Label
1    128027
0     97718
Name: count, dtype: int64
Feature matix: (225745, 78)


np.int64(4)

In [13]:
#Features being converted to numeric data
x = x.apply(pd.to_numeric, errors="coerce") #coerce will set invalid parsing to NaN

#dropping rows with invalid values
x = x.replace([np.inf, -np.inf], np.nan)

mask = x.notna().all(axis=1)
x = x[mask]
y = y[mask]

x.isna().sum().sum() #should be 0 now

np.int64(0)

In [14]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print("X train: ", X_train.shape)
print("X test: ",  X_test.shape)
print("Y train distro: ", y_train.value_counts(normalize=True))
print("Y test distro: ", y_test.value_counts(normalize=True))

X train:  (180568, 78)
X test:  (45143, 78)
Y train distro:  Label
1    0.56721
0    0.43279
Name: proportion, dtype: float64
Y test distro:  Label
1    0.567198
0    0.432802
Name: proportion, dtype: float64


In [15]:
#Model Creation
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix

model = RandomForestClassifier(
    n_estimators= 100,
    random_state= 42,
    n_jobs= -1
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

#Confusion Matrix and Classification Report
con_matrix = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=["Safe", "Malicious"])
print("Confusion Matrix: ", con_matrix)
print("Classification Report: ", report)

Confusion Matrix:  [[19536     2]
 [    3 25602]]
Classification Report:                precision    recall  f1-score   support

        Safe       1.00      1.00      1.00     19538
   Malicious       1.00      1.00      1.00     25605

    accuracy                           1.00     45143
   macro avg       1.00      1.00      1.00     45143
weighted avg       1.00      1.00      1.00     45143

